In [1]:
import numpy as np
from numpy import linalg as LA
import matplotlib.mlab as mlab
import matplotlib.pyplot as plt
import scipy
 
from scipy import linalg
from scipy import stats
from pylab import * 
%matplotlib inline
from scipy.optimize import curve_fit

In [2]:
def PR_ens(vt):
    vPR=np.sum(vt**4,axis=1)
    vPR=1/vPR
    vPRmn=vPR.mean(axis=0)
    return vPRmn

In [3]:
def spectr_contrib_Ens(v1t,vt,nmat,n,t):
    #que tanto contribuye el espectro original con los evectores de PM
    sumarray=[]
    for k in xrange(nmat):
        suma=[]
        for i in xrange(n):
            sumvv=0
            for j in xrange(n-t+1,n):
                v1v=np.dot(v1t[k,:,i],vt[k,:,j])
                sumvv+=(v1v)**2  
            suma.append(sumvv)
        sumarray.append(suma)
    sumarray=np.asarray(sumarray)
    sumn=sumarray.mean(axis=0)    
    return sumn  

In [4]:
def map_level(f, item, level):
    if level == 0:
        return f(item)
    else:
        return [map_level(f, i, level - 1) for i in item]

In [5]:
np.random.seed(9)

In [6]:
nummat=50
N=500
T=200
#e=.01 
#q=1+e 
#corr1=0.9
#corr2=0.3
#k1=50
#k2=N-k1
#W=[]
#q1=q+1
cor1parmt=[.8,.1,.5]
cor2parmt=[.1,.8,.5]
k1parmt=[100,250,300]
qs=[.005,.01,.05,.1,.15,.2]

In [8]:
for ees in qs:
    e=ees
    q=1+e
    q1=1+q
    for item in k1parmt:
        k1=item
        for valor in xrange(len(cor1parmt)):
            corr1=cor1parmt[valor]
            corr2=cor2parmt[valor]
            C=np.zeros((N,N))
            Cq=np.zeros((N,N))
            Corr=np.zeros((N,N))
            Corrsqr=np.zeros((N,N))
            wtotal=np.zeros((nummat,N))
            w1total=np.zeros((nummat,N))
            vtotal=np.zeros(((nummat,N,N)))   
            v1total=np.zeros(((nummat,N,N)))   
            ###  matriz de Xi corr ###
            for i in xrange(k1):
                   for j in xrange(i,k1):
                       if i==j:
                           Corr[i][j]=1
                       else:
                           Corr[i][j]=corr1
                           Corr[j][i]=Corr[i][j]
            for i in xrange(k1,N):
                   for j in xrange(i,N):
                       if i==j:
                           Corr[i][j]=1
                       else:
                           Corr[i][j]=corr2
                           Corr[j][i]=Corr[i][j]
                        
            print "done"                 
            #----------matriz Xi a la 1/2                
            Corrsqr=np.real(scipy.linalg.sqrtm(Corr))
            
            #--------haciendo el ensemble: 
            for matriz in xrange(nummat):
                #### matriz wishart ####

                M1 = 1 * np.random.randn(N, T) + 0	
                #print M1

                ####### normalizando el renglon ######
                M1=[(s-s.mean())/s.std() for s in M1]
                M1=np.asarray(M1)

                #------ haciendo la matriz de corr(wishart correlacionado)----- 

                CorrsqrM1=np.dot(Corrsqr,M1)
                ###normalizando Xi corr por M1
                #CorrsqrM1=[(s-s.mean())/s.std() for s in CorrsqrM1]
                #CorrsqrM1=np.asarray(CorrsqrM1)
                ###
                M2Corrsqr=np.transpose(CorrsqrM1)
                C=np.dot(CorrsqrM1,M2Corrsqr)
                C=C/T
                
                ###### --diagonalizando C--- #########
                w, v = LA.eigh(C)
                #W=w
                wtotal[matriz]=w
                #print "wtotal done", matriz
                vtotal[matriz]=v
                #print "vtotal done", matriz
                    
                ##### ----power mapping--- #####

                Cq=map_level(lambda x: abs(x)**(q1)/x, C, 2)
                Cq=np.asarray(Cq)
                ### ---diagonalizando Cq----###
                
                w1, v1 = LA.eigh(Cq)
                
                v1total[matriz]=v1
                w1total[matriz]=w1
                                    
            ######### calculando el PR #########

            
            ##con PM            
            v1IPR=PR_ens(v1total)           
            plt.plot(v1IPR)
            #plt.title("PR-PM:%dx%d k1=%d cor1=%.2f cor2=%.2f q=%.3f" % (N,T, k1,corr1,corr2,q),fontsize=16)
            titulo="PR-PM:%dx%d,k1=%d,cor1=%.2f,cor2=%.2f,q=%.3f.png" %(N,T, k1,corr1,corr2,q)
            plt.ylabel("$PR(VP^j)$",fontsize=14)
            plt.xlabel("$VP^j$",fontsize=14)
            savefig(titulo, bbox_inches='tight')
            plt.clf()
            plt.close()
            
            ##sin PM
            vIPR=PR_ens(vtotal)                        
            plt.plot(vIPR)
            #plt.title("PR:%dx%d k1=%d cor1=%.2f cor2=%.2f q=%.3f" % (N,T, k1,corr1,corr2,q),fontsize=16)
            titulo="PR:%dx%d,k1=%d,cor1=%.2f,cor2=%.2f,q=%.3f.png" %(N,T, k1,corr1,corr2,q)
            plt.ylabel("$PR(V^j)$",fontsize=14)
            plt.xlabel("$V^j$",fontsize=14)
            savefig(titulo, bbox_inches='tight')
            plt.clf()
            plt.close()        
            ##### que tanto contribuye el el espectro original en los evectores con PM
            sumamn=spectr_contrib_Ens(v1total,vtotal,nummat,N,T)
            ##primeros N-T evect
            plt.semilogy(sumamn[:])
            #plt.title("dot(espectro orign):%dx%d k1=%d cor1=%.2f cor2=%.2f q=%.3f" % (N,T,k1,corr1,corr2,q),fontsize=16)
            titulo="dot:%dx%d,k1=%d,cor1=%.2f,cor2=%.2f,q=%.3f.png" %(N,T, k1,corr1,corr2,q)
            plt.xlabel("$VP^j$",fontsize=14)
            plt.ylabel("$OSC(VP^j)$",fontsize=14)
            savefig(titulo, bbox_inches='tight')
            plt.clf()
            plt.close()
            ## grafica eel espectro original
            #plt.semilogy(sumamn[:N-T])
            #plt.title("dot(espectro orign):%dx%d k1=%d cor1=%.2f cor2=%.2f q=%.3f" % (N,T,k1,corr1,corr2,q),fontsize=16)
            #titulo="dotEsEM:%dx%d,k1=%d,cor1=%.2f,cor2=%.2f,q=%.3f.png" %(N,T, k1,corr1,corr2,q)
            #plt.xlabel("$VP^j$",fontsize=14)
            #plt.ylabel("$OSC(VP^j)$",fontsize=14)
            #savefig(titulo, bbox_inches='tight')
            #plt.clf()
            #plt.close()
            
            ##guardando evector y evalores
            
            vtotal=[ s.flatten() for s in vtotal]
            vtotal=np.asarray(vtotal)
            nombre="vector:%dx%d,k1=%d,cor1=%.2f,cor2=%.2f,q=%.3f.dat" %(N,T, k1,corr1,corr2,q)
            np.savetxt(nombre,vtotal, delimiter=" ",fmt="%2.8f")
            
            nombre="vectorPM:%dx%d,k1=%d,cor1=%.2f,cor2=%.2f,q=%.3f.dat" %(N,T, k1,corr1,corr2,q)            
            v1total=[ s.flatten() for s in v1total]
            v1total=np.asarray(v1total)
            np.savetxt(nombre,v1total, delimiter=" ",fmt="%2.8f")
            
            nombre="valor:%dx%d,k1=%d,cor1=%.2f,cor2=%.2f,q=%.3f.dat" %(N,T, k1,corr1,corr2,q)
            np.savetxt(nombre,wtotal, delimiter=" ",fmt="%2.8f")
            nombre="valorPM:%dx%d,k1=%d,cor1=%.2f,cor2=%.2f,q=%.3f.dat" %(N,T, k1,corr1,corr2,q)                     
            np.savetxt(nombre,w1total, delimiter=" ",fmt="%2.8f")
                                                    
                                        

done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
